In [1]:
import tensorflow as tf
import os
import numpy as np
# from osgeo import gdal, osr
# import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
from transformers import Mask2FormerForUniversalSegmentation, AutoConfig, Mask2FormerConfig
import torch.nn as nn

2026-02-09 10:47:01.932384: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
import torch
import torch.nn as nn
from transformers import SegformerForSemanticSegmentation

class SegFormerSegmentor(nn.Module):
    def __init__(self, num_classes, num_channels=4):
        super(SegFormerSegmentor, self).__init__()
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0",
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        # Adjust input conv layer for different channel numbers (default is 3 for RGB)
        if num_channels != 3:
            old_conv = self.model.segformer.encoder.patch_embeddings[0].proj
            new_conv = nn.Conv2d(num_channels,
                                 old_conv.out_channels,
                                 kernel_size=old_conv.kernel_size,
                                 stride=old_conv.stride,
                                 padding=old_conv.padding,
                                 bias=old_conv.bias is not None)
            # Copy weights for first 3 channels if available, random init for the rest
            with torch.no_grad():
                new_conv.weight[:, :3, :, :] = old_conv.weight
                if num_channels > 3:
                    nn.init.xavier_uniform_(new_conv.weight[:, 3:, :, :])
            self.model.segformer.encoder.patch_embeddings[0].proj = new_conv

    def forward(self, x):
        """
        x: torch.Tensor with shape [B, C, H, W]
        """
        outputs = self.model(x)
        logits = outputs.logits  # [B, num_classes, H/4, W/4] (usually smaller resolution)
        # Upsample back to input resolution
        logits = nn.functional.interpolate(
            logits,
            size=(x.shape[2], x.shape[3]),
            mode="bilinear",
            align_corners=False
        )
        # logits = torch.relu(logits)
        return logits


# Example usage
if __name__ == "__main__":
    model = SegFormerSegmentor(num_classes=10, num_channels=4)
    dummy_input = torch.randn(2, 4, 256, 256)  # [batch, channels, height, width]
    output = model(dummy_input)
    print(output.shape)  # torch.Size([2, 10, 256, 256])

/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode

torch.Size([2, 10, 256, 256])


In [4]:
def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)*0.0001

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)*0.0000275-0.2

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label[..., 0]

        @tf.function
        def _augment_function(lres_img, hres_img, label):
            # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
            if tf.rank(label) == 2:
                label = tf.expand_dims(label, axis=-1)
        
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # ---- Random horizontal flip ----
            do_flip_lr = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_lr,
                            lambda: tf.image.flip_left_right(label),
                            lambda: label)
        
            # ---- Random vertical flip ----
            do_flip_ud = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_ud,
                            lambda: tf.image.flip_up_down(label),
                            lambda: label)
        
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
            label = tf.squeeze(label, axis=-1)
        
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

# filenames = '/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords'
# ds = input_pipeline_downstream_sr(filenames, 10, 0, 200, is_shuffle=True, is_train=True, is_repeat=True)


# for lres_batch, hres_batch, label in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 3, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))
#         hres_img = np.transpose(hres_batch[i].numpy(), (1, 2, 0))
#         label_img = label[i].numpy()

#         axes[0].imshow(lres_img[:, :, [0,3,2]]*3)
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img[:, :, 3:0:-1]*3)
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         axes[2].imshow(label_img)
#         axes[2].set_title('label')
#         axes[2].axis('off')

#         plt.show()

#     break

In [5]:
def evaluate_unet(model, test_ds, img_size, label_size, class_num, start_class, device):
    model.eval()

    correct_pixels = 0
    total_pixels = 0

    if class_num == 2:
        TP = total_gt = total_pred = 0
    else:
        TP_per_class = [0] * class_num
        gt_per_class = [0] * class_num
        pred_per_class = [0] * class_num

    with torch.no_grad():
        for images, images2, labels in test_ds:
            if img_size == 64:
                images = torch.from_numpy(images.numpy().astype("float32")).to(device)
                images = F.interpolate(
                    images,
                    size=(img_size, img_size),
                    mode="bilinear",
                    align_corners=False
                )
                logits = model(images)
            if img_size == 1024:
                images2 = torch.from_numpy(images2.numpy().astype("float32")).to(device)
                images2 = F.interpolate(
                    images2,
                    size=(img_size, img_size),
                    mode="bilinear",
                    align_corners=False
                )
                logits = model(images2)
                
            logits = F.interpolate(
                logits,
                size=(label_size, label_size),
                mode="bilinear",
                align_corners=False
            )
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            labels = labels.numpy()

            correct_pixels += np.sum(preds == labels)
            total_pixels += labels.size

            if class_num == 2:
                TP += np.sum((preds == 1) & (labels == 1))
                total_gt += np.sum(labels == 1)
                total_pred += np.sum(preds == 1)
            else:
                for i in range(start_class, class_num):
                    gt_mask = labels == i
                    pred_mask = preds == i
                    TP_per_class[i] += np.sum(gt_mask & pred_mask)
                    gt_per_class[i] += np.sum(gt_mask)
                    pred_per_class[i] += np.sum(pred_mask)

    acc = correct_pixels / total_pixels

    if class_num == 2:
        mf1 = 2 * TP / (total_gt + total_pred) if (total_gt + total_pred) > 0 else 0
    else:
        f1_sum, valid = 0, 0
        for i in range(start_class, class_num):
            denom = gt_per_class[i] + pred_per_class[i]
            if denom > 0:
                f1_sum += 2 * TP_per_class[i] / denom
                valid += 1
        mf1 = f1_sum / valid if valid > 0 else 0

    return acc, mf1

In [6]:
import pandas as pd

def unet_run(
    lres_size,
    hres_size,
    hres_size_4x,
    label_size,
    num_sample,
    num_training,
    class_num,
    start_class,
    finetune_tfrecords,
    unet_s2_save_path_tpl,     # e.g. "..._run{}.pth"
    unet_naip_save_path_tpl    # e.g. "..._run{}.pth"
):
    device = "cuda"
    num_test = num_sample - num_training
    runs = [1, 2, 3]

    mf1_rows = {}
    acc_rows = {}

    test_ds = input_pipeline_downstream_sr(
        finetune_tfrecords,
        batch_size=4,
        skip=num_training,
        take=num_test,
        is_shuffle=False,
        is_train=False,
        is_repeat=False
    )

    for run in runs:
        # ---- S2 ----
        print('Load: ', unet_s2_save_path_tpl.format(run).replace(".pth", "_best.pth"))
        model_s2 = SegFormerSegmentor(num_classes=class_num, num_channels=7).to(device)
        model_s2.load_state_dict(
            torch.load(unet_s2_save_path_tpl.format(run).replace(".pth", "_best.pth"), weights_only=True)
        )

        acc_s2, mf1_s2 = evaluate_unet(
            model_s2, test_ds, 64, label_size, class_num, start_class, device
        )

        # ---- NAIP ----
        print('Load: ', unet_naip_save_path_tpl.format(run).replace(".pth", "_best.pth"))
        model_naip = SegFormerSegmentor(num_classes=class_num, num_channels=7).to(device)
        model_naip.load_state_dict(
            torch.load(unet_naip_save_path_tpl.format(run).replace(".pth", "_best.pth"), weights_only=True)
        )

        acc_naip, mf1_naip = evaluate_unet(
            model_naip, test_ds, 1024, label_size, class_num, start_class, device
        )

        row = f"run{run}"
        mf1_rows[row] = {"S2": mf1_s2, "NAIP": mf1_naip}
        acc_rows[row] = {"S2": acc_s2, "NAIP": acc_naip}

    # ---- DataFrames ----
    mf1_df = pd.DataFrame.from_dict(mf1_rows, orient="index")
    acc_df = pd.DataFrame.from_dict(acc_rows, orient="index")

    # ---- Average row ----
    mf1_df.loc["Average"] = mf1_df.mean(axis=0)
    acc_df.loc["Average"] = acc_df.mean(axis=0)

    return mf1_df, acc_df

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1065
num_training = 852
class_num=2
start_class=0
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_M2_Segformer_run{}.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_L8_Segformer_run{}.pth'

mf1_df, acc_df = unet_run(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         start_class,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path
        )


mf1_df.to_excel("M2L8_River_original_Segformer_mf1.xlsx")
acc_df.to_excel("M2L8_River_original_Segformer_accuracy.xlsx")

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1687
num_training = 1350
class_num=11
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_M2_Segformer_run{}.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_L8_Segformer_run{}.pth'
start_class = 0


mf1_df, acc_df = unet_run(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         start_class,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path
        )

mf1_df.to_excel("M2L8_Urban_original_Segformer_mf1.xlsx")
acc_df.to_excel("M2L8_Urban_original_Segformer_accuracy.xlsx")

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 762
num_training = 610
class_num=5
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_M2_Segformer_run{}.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_L8_Segformer_run{}.pth'
start_class = 0

mf1_df, acc_df = unet_run(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         start_class,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path
        )

mf1_df.to_excel("M2L8_CDL_original_Segformer_mf1.xlsx")
acc_df.to_excel("M2L8_CDL_original_Segformer_accuracy.xlsx")

# Regression

In [11]:
def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)*0.0001

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)*0.0000275-0.2

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            label = tf.cast(label, tf.float32)*value_multiplier
            return lres, hres, label[..., 0]

        @tf.function
        def _augment_function(lres_img, hres_img, label):
            # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
            if tf.rank(label) == 2:
                label = tf.expand_dims(label, axis=-1)
        
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # ---- Random horizontal flip ----
            do_flip_lr = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_lr,
                               lambda: tf.image.flip_left_right(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_lr,
                            lambda: tf.image.flip_left_right(label),
                            lambda: label)
        
            # ---- Random vertical flip ----
            do_flip_ud = tf.random.uniform([]) > 0.5
            lres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(lres_img),
                               lambda: lres_img)
            hres_img = tf.cond(do_flip_ud,
                               lambda: tf.image.flip_up_down(hres_img),
                               lambda: hres_img)
            label = tf.cond(do_flip_ud,
                            lambda: tf.image.flip_up_down(label),
                            lambda: label)
        
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
            label = tf.squeeze(label, axis=-1)
        
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

In [12]:
def evaluate_unet(model, test_ds, img_size, label_size, class_num, device):
    model.eval()

    mae_total = 0.0
    rmse_total = 0.0
    pixel_count = 0

    with torch.no_grad():
        for images, images2, labels in test_ds:
            if img_size == 64:
                images = torch.from_numpy(images.numpy().astype("float32")).to(device)
                images = F.interpolate(
                    images,
                    size=(img_size, img_size),
                    mode="bilinear",
                    align_corners=False
                )
                logits = model(images)
            if img_size == 1024:
                images2 = torch.from_numpy(images2.numpy().astype("float32")).to(device)
                images2 = F.interpolate(
                    images2,
                    size=(img_size, img_size),
                    mode="bilinear",
                    align_corners=False
                )
                logits = model(images2)
                
            preds = F.interpolate(logits, size=(label_size, label_size), mode='nearest')
            preds = preds.squeeze(1).cpu().numpy()  # [B, H, W] for regression
    
            labels = labels.numpy()  # [B, H, W] if channel dim exists
    
            abs_error = np.abs(preds - labels)
            # print(abs_error.shape)
            sq_error = (preds - labels) ** 2
    
            mae_total += np.sum(abs_error)
            rmse_total += np.sum(sq_error)
            pixel_count += np.prod(labels.shape)
    
        mae = mae_total / pixel_count
        rmse = np.sqrt(rmse_total / pixel_count)

    return mae, rmse

In [13]:
import pandas as pd

def unet_run(
    lres_size,
    hres_size,
    hres_size_4x,
    label_size,
    num_sample,
    num_training,
    class_num,
    finetune_tfrecords,
    unet_s2_save_path_tpl,     # e.g. "..._run{}.pth"
    unet_naip_save_path_tpl    # e.g. "..._run{}.pth"
):
    device = "cuda"
    num_test = num_sample - num_training
    runs = [1, 2, 3]

    mf1_rows = {}
    acc_rows = {}

    test_ds = input_pipeline_downstream_sr(
        finetune_tfrecords,
        batch_size=4,
        skip=num_training,
        take=num_test,
        is_shuffle=False,
        is_train=False,
        is_repeat=False
    )

    for run in runs:
        # ---- S2 ----
        print('Load: ', unet_s2_save_path_tpl.format(run).replace(".pth", "_best.pth"))
        model_s2 = SegFormerSegmentor(num_classes=class_num, num_channels=7).to(device)
        model_s2.load_state_dict(
            torch.load(unet_s2_save_path_tpl.format(run).replace(".pth", "_best.pth"), weights_only=True)
        )

        acc_s2, mf1_s2 = evaluate_unet(
            model_s2, test_ds, 64, label_size, class_num, device
        )

        # ---- NAIP ----
        print('Load: ', unet_naip_save_path_tpl.format(run).replace(".pth", "_best.pth"))
        model_naip = SegFormerSegmentor(num_classes=class_num, num_channels=7).to(device)
        model_naip.load_state_dict(
            torch.load(unet_naip_save_path_tpl.format(run).replace(".pth", "_best.pth"), weights_only=True)
        )

        acc_naip, mf1_naip = evaluate_unet(
            model_naip, test_ds, 1024, label_size, class_num, device
        )

        row = f"run{run}"
        mf1_rows[row] = {"S2": mf1_s2, "NAIP": mf1_naip}
        acc_rows[row] = {"S2": acc_s2, "NAIP": acc_naip}

    # ---- DataFrames ----
    mf1_df = pd.DataFrame.from_dict(mf1_rows, orient="index")
    acc_df = pd.DataFrame.from_dict(acc_rows, orient="index")

    # ---- Average row ----
    mf1_df.loc["Average"] = mf1_df.mean(axis=0)
    acc_df.loc["Average"] = acc_df.mean(axis=0)

    return mf1_df, acc_df

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 755
num_training = 604
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_M2_Segformer_run{}.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_L8_Segformer_run{}.pth'
value_multiplier = 0.0001

mf1_df, acc_df = unet_run(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path
        )

mf1_df.to_excel("M2L8_GPP_original_Segformer_mae.xlsx")
acc_df.to_excel("M2L8_GPP_original_Segformer_rmse.xlsx")

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 1408
num_training = 1126
num_test = num_sample-num_training
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']
unet_s2_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_M2_Segformer_run{}.pth'
unet_naip_save_path = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_L8_Segformer_run{}.pth'
value_multiplier = 0.1

mf1_df, acc_df = unet_run(lres_size ,
         hres_size ,
         hres_size_4x ,
         label_size ,
         num_sample,
         num_training ,
         class_num,
         finetune_tfrecords,
         unet_s2_save_path,
         unet_naip_save_path
        )

mf1_df.to_excel("M2L8_CHM_original_Segformer_mae.xlsx")
acc_df.to_excel("M2L8_CHM_original_Segformer_rmse.xlsx")